# CodeCompass: Fine-Tuning for Code Explanation

This notebook fine-tunes Qwen 2.5 7B to produce better code explanations.

**Training Data:**
- **Magicoder-OSS-Instruct 75k** — Code → Explanation pairs (parsed from solutions)

**Why not tool calling?**
- Base Qwen 2.5 7B already achieves 95%+ tool selection accuracy
- Fine-tuning focuses on what base model needs help with: concise, clear explanations

**Artifcacts:**
- A LoRA adapter for code explanation (hot-swappable)
  - Must use llama.cpp afterwards to convert to gguf since we don't want to merge adapter to model and have user download two seperate models

---

## 1️⃣ Environment Setup

In [1]:
# Detect environment
import os
import sys

IN_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None
IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False

print(f"Running in: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local'}")

# Set output directory
if IN_KAGGLE:
    OUTPUT_DIR = "/kaggle/working/codecompass-model"
    DRIVE_BACKUP = None
elif IN_COLAB:
    OUTPUT_DIR = "/content/codecompass-model"
    DRIVE_BACKUP = "/content/drive/MyDrive/codecompass-model"
else:
    OUTPUT_DIR = "./outputs/codecompass-model"
    DRIVE_BACKUP = None

print(f"Output directory: {OUTPUT_DIR}")

Running in: Colab
Output directory: /content/codecompass-model


In [2]:
# Install dependencies (~3-5 minutes)

if IN_KAGGLE:
    !pip install -q "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
elif IN_COLAB:
    !pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
else:
    !pip install -q "unsloth @ git+https://github.com/unslothai/unsloth.git"

!pip install -q datasets huggingface_hub rich

# # Pin compatible versions (fixes 'int' object has no attribute 'mean' error)
!pip install -q --no-deps "trl==0.12.2" "peft>=0.14.0" accelerate bitsandbytes
# !pip install -q xformers==0.0.28.post3 --no-deps
# !pip install -q trl==0.12.2 peft==0.14.0 accelerate==1.2.1 bitsandbytes==0.45.0 --no-deps

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.7/290.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 24.0 MB/s eta 0:00:00


In [3]:
# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected! Training will be extremely slow.")
    sys.exit(1)

PyTorch: 2.9.0+cu126
CUDA available: True
GPU: Tesla T4
Memory: 15.8 GB


In [4]:
# Optional: Mount Google Drive (Colab only)
if IN_COLAB and DRIVE_BACKUP:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_BACKUP, exist_ok=True)
    print(f"Drive mounted. Backups: {DRIVE_BACKUP}")

Mounted at /content/drive
Drive mounted. Backups: /content/drive/MyDrive/codecompass-model


## 2️⃣ Download & Process Magicoder Dataset

In [5]:
from datasets import load_dataset
import json
import re
import random
from typing import Optional, Dict
from pathlib import Path

print("Downloading Magicoder-OSS-Instruct-75K...")
magicoder_dataset = load_dataset("ise-uiuc/Magicoder-OSS-Instruct-75K", split="train")
print(f"✓ Downloaded {len(magicoder_dataset):,} examples")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/314 [00:00<?, ?B/s]

data-oss_instruct-decontaminated.jsonl:   0%|          | 0.00/203M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/75197 [00:00<?, ? examples/s]

✓ Downloaded 75,197 examples


In [6]:
def parse_magicoder_to_explanation(item: dict) -> Optional[Dict]:
    """Convert Magicoder with stricter quality filters."""

    solution = item.get("solution", "")

    # Extract code block
    code_match = re.search(
        r"```(?:python)?\s*(.*?)```",
        solution,
        re.DOTALL | re.IGNORECASE
    )
    if not code_match:
        return None

    code = code_match.group(1).strip()
    explanation = solution[code_match.end():].strip()
    explanation_lower = explanation.lower()

    # Skip meta/assessment text (not actual explanations)
    meta_patterns = [
        "this problem assesses",
        "this problem tests",
        "the candidate",
        "this exercise",
        "the task is to",
        "you need to implement",
    ]
    if any(pattern in explanation_lower for pattern in meta_patterns):
        return None

    # Skip shallow explanations
    shallow_patterns = [
        "the function returns",
        "the code above",
        "when the above code is executed",
        "this will produce the following output",
        "the completed function",
        "the corrected function",
        "is implemented with the required",
        "is implemented with the specified",
        "the function takes the input",
        "the missing part",
        "the problematic parts",
        "the corrected version",
        "when the provided code",
        "the output will be",
        "correctly implements",
        "correctly calculating",
    ]
    if any(pattern in explanation_lower for pattern in shallow_patterns):
        return None

    # Must have actual explanation content, not just restatement
    insight_verbs = [
        "ensures", "allows", "enables", "prevents", "avoids",
        "handles", "manages", "optimizes", "improves",
        "because", "since", "therefore", "thus",
        "the key", "the reason", "this approach", "this pattern",
        "this technique", "this is useful", "this makes",
        "efficiently", "safely", "correctly"
    ]
    has_insight = any(verb in explanation_lower for verb in insight_verbs)

    if not has_insight:
        return None

    # Truncate very long explanations
    if len(explanation) > 1500:
        explanation = explanation[:1500]

    return {
        "messages": [
            {
                "role": "system",
                "content": "Explain this code concisely. Focus on what it does, why it's designed this way, and any notable patterns or techniques."
            },
            {"role": "user", "content": code},
            {"role": "assistant", "content": explanation}
        ]
    }


print("Processing dataset...")

Processing dataset...


In [7]:
# Process all examples
examples = []
skipped = {"no_code_block": 0, "not_python": 0, "too_short": 0, "wrong_lang": 0, "has_solution_word": 0}

for item in magicoder_dataset:
    # Filter: Python only (Magicoder has 'lang' field)
    if item.get("lang", "").lower() != "python":
        skipped["wrong_lang"] += 1
        continue

    # Filter: Skip if solution contains "solution" (often incomplete examples)
    solution_text = item.get("solution", "")
    if "solution" in solution_text.lower():
        skipped["has_solution_word"] += 1
        continue

    parsed = parse_magicoder_to_explanation(item)
    if parsed:
        examples.append(parsed)

print(f"\n✓ Generated {len(examples):,} training examples")
print(f"\nSkipped:")
for reason, count in skipped.items():
    if count > 0:
        print(f"  {reason}: {count:,}")


✓ Generated 1,454 training examples

Skipped:
  wrong_lang: 36,913
  has_solution_word: 18,282


In [8]:
MAX_EXAMPLES = 2000

if len(examples) > MAX_EXAMPLES:
    random.seed(42)
    examples = random.sample(examples, MAX_EXAMPLES)
    print(f"Sampled down to {len(examples):,} examples")

# Shuffle
random.shuffle(examples)
print(f"\nFinal dataset size: {len(examples):,} examples")


Final dataset size: 1,454 examples


In [9]:
# Preview an example
print("=" * 70)
print("EXAMPLE:")
print("=" * 70)
ex = examples[0]
print(f"\n[CODE]\n{ex['messages'][1]['content'][:500]}...")
print(f"\n[EXPLANATION]\n{ex['messages'][2]['content'][:500]}...")

EXAMPLE:

[CODE]
def calculate_word_statistics(word_list):
    print("\nCalculating word statistics ...")
    word_count = len(word_list)
    unique_words = set(word_list)
    unique_word_count = len(unique_words)
    word_frequency = {word: word_list.count(word) for word in unique_words}

    print(f"Total words: {word_count}")
    print(f"Unique words: {unique_word_count}")
    print("Word frequency:")
    for word, frequency in word_frequency.items():
        print(f"- {word}: {frequency}")

# Example usage
w...

[EXPLANATION]
This Python program defines a function `calculate_word_statistics` that takes a list of words as input and calculates the required statistics. It uses built-in functions and data structures to efficiently compute the total number of words, the number of unique words, and the frequency of each word in the list. Finally, it prints the calculated statistics in the specified format....


## 3️⃣ Train/Validation Split

In [10]:
# 90/10 split
split_idx = int(len(examples) * 0.9)
train_data = examples[:split_idx]
val_data = examples[split_idx:]

print(f"Training examples:   {len(train_data):,}")
print(f"Validation examples: {len(val_data):,}")

# Save validation data for later evaluation
val_export_path = Path(OUTPUT_DIR) / "val_data.json"
os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(val_export_path, "w") as f:
    json.dump(val_data, f, indent=2)
print(f"\n✓ Validation data saved to: {val_export_path}")

Training examples:   1,308
Validation examples: 146

✓ Validation data saved to: /content/codecompass-model/val_data.json


## 4️⃣ Load Model

In [ ]:
# # Clear GPU memory
# import gc
# import torch

# # Delete model and trainer if they exist
# for var in ['model', 'trainer', 'tokenizer']:
#     if var in dir():
#         exec(f'del {var}')

# gc.collect()
# torch.cuda.empty_cache()

# print(f"GPU memory freed: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

In [11]:
from unsloth import FastLanguageModel

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

print(f"Loading {MODEL_NAME}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=torch.float16,
)

print(f"✓ Model loaded")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading Qwen/Qwen2.5-7B-Instruct...
==((====))==  Unsloth 2025.12.9: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.16G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

✓ Model loaded


In [12]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✓ LoRA added: {trainable:,} trainable ({100*trainable/total:.2f}%)")

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.12.9 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


✓ LoRA added: 161,480,704 trainable (3.20%)


## 5️⃣ Prepare Dataset for Training

In [13]:
from datasets import Dataset

def format_for_trainer(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

train_dataset = train_dataset.map(format_for_trainer, remove_columns=train_dataset.column_names)
val_dataset = val_dataset.map(format_for_trainer, remove_columns=val_dataset.column_names)

print(f"Train: {len(train_dataset):,} | Val: {len(val_dataset):,}")

Map:   0%|          | 0/1308 [00:00<?, ? examples/s]

Map:   0%|          | 0/146 [00:00<?, ? examples/s]

Train: 1,308 | Val: 146


In [14]:
# Preview formatted example
print("Formatted example (first 500 chars):")
print(train_dataset[0]["text"][:500])

Formatted example (first 500 chars):
<|im_start|>system
Explain this code concisely. Focus on what it does, why it's designed this way, and any notable patterns or techniques.<|im_end|>
<|im_start|>user
def calculate_word_statistics(word_list):
    print("\nCalculating word statistics ...")
    word_count = len(word_list)
    unique_words = set(word_list)
    unique_word_count = len(unique_words)
    word_frequency = {word: word_list.count(word) for word in unique_words}

    print(f"Total words: {word_count}")
    print(f"Unique w


## 6️⃣ Training

In [ ]:
print(len(train_dataset), len(val_dataset))


In [ ]:
print(train_dataset.column_names)


In [15]:
from transformers import TrainingArguments
from trl import SFTTrainer
import psutil
import builtins

builtins.psutil = psutil  # <- makes psutil visible to the trainer

batch_size = 1          # Reduce from 2
grad_accum = 16         # Increase to maintain effective batch size

effective_batch = batch_size * grad_accum
steps_per_epoch = len(train_dataset) // effective_batch

print(f"Effective batch size: {effective_batch}")
print(f"Steps per epoch: {steps_per_epoch:,}")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    optim="adamw_8bit",
    weight_decay=0.01,
    fp16=True,
    save_strategy="steps",
    save_steps=max(100, steps_per_epoch // 2),
    save_total_limit=2,
    logging_steps=25,
    eval_strategy="no",       # Disable eval to save memory
    # eval_strategy="steps",
    eval_steps=max(100, steps_per_epoch // 2),
    report_to="none",
    seed=42,
    gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,
    eval_packing=False,

)

Effective batch size: 16
Steps per epoch: 81


/content/unsloth_compiled_cache/UnslothSFTTrainer.py:664: UserWarning: You passed a `packing` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/content/unsloth_compiled_cache/UnslothSFTTrainer.py:669: UserWarning: You passed a `eval_packing` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/content/unsloth_compiled_cache/UnslothSFTTrainer.py:752: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/content/unsloth_compiled_cache/UnslothSFTTrainer.py:780: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/146 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


In [16]:
# Check for checkpoint to resume from
resume_from = None
checkpoint_dir = Path(OUTPUT_DIR)

if checkpoint_dir.exists():
    checkpoints = [d for d in checkpoint_dir.iterdir()
                   if d.is_dir() and d.name.startswith("checkpoint-")]
    if checkpoints:
        latest = max(checkpoints, key=lambda x: int(x.name.split("-")[1]))
        resume_from = str(latest)
        print(f"⏩ Resuming from: {resume_from}")
    else:
        print("🆕 Starting fresh")
else:
    print("🆕 Starting fresh")

🆕 Starting fresh


In [17]:
# TRAIN!
print("=" * 60)
print("🚀 STARTING TRAINING")
print("=" * 60)
print(f"Training examples: {len(train_dataset):,}")
print(f"Epochs: 2")
print(f"Learning rate: 2e-4")
print(f"Checkpoints save to: {OUTPUT_DIR}")
print("\nIf session times out, re-run this cell to resume!\n")

if resume_from:
    trainer.train(resume_from_checkpoint=resume_from)
else:
    trainer.train()

print("\n" + "=" * 60)
print("✅ TRAINING COMPLETE!")
print("=" * 60)

The model is already on multiple devices. Skipping the move to device specified in `args`.


🚀 STARTING TRAINING
Training examples: 1,308
Epochs: 2
Learning rate: 2e-4
Checkpoints save to: /content/codecompass-model

If session times out, re-run this cell to resume!



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 153 | Num Epochs = 2 | Total steps = 20
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 161,480,704 of 7,777,097,216 (2.08% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss



✅ TRAINING COMPLETE!


## 7️⃣ Save LoRA Adapter (Separate from Base)

In [18]:
LORA_ADAPTER_DIR = f"{OUTPUT_DIR}/lora_adapter"

# Save LoRA adapter only (NOT merged with base)
model.save_pretrained(LORA_ADAPTER_DIR)
tokenizer.save_pretrained(LORA_ADAPTER_DIR)

print(f"✓ LoRA adapter saved to: {LORA_ADAPTER_DIR}")
print(f"\nThis adapter can be hot-swapped with LangGraph.")
print(f"Base model (qwen2.5:7b) remains unchanged for tool calling.")

✓ LoRA adapter saved to: /content/codecompass-model/lora_adapter

This adapter can be hot-swapped with LangGraph.
Base model (qwen2.5:7b) remains unchanged for tool calling.


In [19]:
# Optional: Backup to Google Drive (Colab)
if IN_COLAB and DRIVE_BACKUP:
    import shutil
    print(f"Backing up to Google Drive...")
    shutil.copytree(LORA_ADAPTER_DIR, f"{DRIVE_BACKUP}/lora_adapter", dirs_exist_ok=True)
    shutil.copy(f"{OUTPUT_DIR}/val_data.json", f"{DRIVE_BACKUP}/val_data.json")
    print(f"✓ Backed up to: {DRIVE_BACKUP}")

Backing up to Google Drive...
✓ Backed up to: /content/drive/MyDrive/codecompass-model


## 8️⃣ Quick Evaluation

In [ ]:
# Enable inference mode
FastLanguageModel.for_inference(model)

def generate_explanation(code: str) -> str:
    messages = [
        {"role": "system", "content": "Explain this code concisely. Focus on what it does, why it's designed this way, and any notable patterns."},
        {"role": "user", "content": code}
    ]

    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

print("✓ Inference ready")

✓ Inference ready


In [ ]:
# Test on sample code
TEST_CODES = [
    '''def retry_with_backoff(func, max_retries=3):
    for attempt in range(max_retries):
        try:
            return func()
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            time.sleep(2 ** attempt)''',

    '''class Singleton:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance''',

    '''def memoize(func):
    cache = {}
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]
    return wrapper''',
]

print("Testing fine-tuned model...\n")

for i, code in enumerate(TEST_CODES, 1):
    print(f"{'='*60}")
    print(f"TEST {i}")
    print(f"{'='*60}")
    print(f"[CODE]\n{code}\n")

    explanation = generate_explanation(code)
    print(f"[EXPLANATION]\n{explanation}\n")

Testing fine-tuned model...

TEST 1
[CODE]
def retry_with_backoff(func, max_retries=3):
    for attempt in range(max_retries):
        try:
            return func()
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            time.sleep(2 ** attempt)

[EXPLANATION]
This function `retry_with_backoff` attempts to execute a given function `func` up to `max_retries` times with exponential backoff between retries. It catches exceptions and waits progressively longer (doubling the wait time each retry) before retrying, until the maximum number of retries is reached or no exception occurs.

- `func`: The function to be executed.
- `max_retries`: Maximum number of retries (default 3).
- Uses `time.sleep()` to introduce delays between retries.
- Retries only if an exception occurs; otherwise, returns immediately.
- Raises the last caught exception if all retries fail.

TEST 2
[CODE]
class Singleton:
    _instance = None
    
    def __new__(cls):


## 9️⃣ Export LoRA Adapter to GGUF

**Important:** We need to export the LoRA adapter separately, NOT merged with the base model.

Unsloth's `save_pretrained_gguf()` merges everything by default. For true hot-swap, we use llama.cpp's `convert_lora_to_gguf.py`.

In [ ]:
# The LoRA adapter is already saved in HuggingFace format at LORA_ADAPTER_DIR
# To convert to GGUF adapter format, use llama.cpp on your local machine

print("=" * 70)
print("LORA ADAPTER EXPORT")
print("=" * 70)
print(f"""
✓ LoRA adapter saved to: {LORA_ADAPTER_DIR}

To convert to GGUF adapter for Ollama hot-swap:

1. Download the lora_adapter folder to your local machine

2. Clone llama.cpp and convert:

   git clone https://github.com/ggerganov/llama.cpp
   cd llama.cpp
   pip install -r requirements.txt

   python convert_lora_to_gguf.py \\
       --base Qwen/Qwen2.5-7B-Instruct \\
       --lora /path/to/lora_adapter \\
       --outfile codecompass-explain-adapter.gguf

3. Create Ollama model with adapter:

   # Create Modelfile
   cat > Modelfile << 'EOF'
   FROM qwen2.5:7b
   ADAPTER ./codecompass-explain-adapter.gguf

   SYSTEM """Explain this code concisely. Focus on what it does, why it's designed this way, and any notable patterns or techniques."""

   PARAMETER temperature 0.1
   PARAMETER stop "<|im_end|>"
   EOF

   ollama create codecompass:explain -f Modelfile

4. Test:
   ollama run codecompass:explain "Explain: def foo(): pass"
""")